# ESTR3114: Multitask CIFAR-100 — scalar joint versus PCGrad

Twenty known-superclass, five-class tasks share a CNN. This is **not ordinary
100-class classification**. Select **Runtime → Change runtime type → GPU** and
run cells in order. Push `new_experiment/` to your repository before cloning.

Assets/downloads remain on Colab's temporary disk. Flat condition folders,
checkpoints, and analysis persist in Google Drive. Model saving is enabled so
final and best-validation errors can be analysed later. Test evaluation is a
separate optional step after inspecting validation results.

## 1. Clone the repository and locate the independent experiment

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import os

REPOSITORY_URL = "https://github.com/marcolo0714-lgtm/ESTR3114_Project"
CHECKOUT = Path("/content/estr3114-project")
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(CHECKOUT)], check=True)
elif not (CHECKOUT / ".git").is_dir():
    raise RuntimeError(f"Existing path is not a Git checkout: {CHECKOUT}")
else:
    print("Reusing checkout. It is not automatically pulled or overwritten.")

project = CHECKOUT / "new_experiment"
assert (project / "1_prepare_assets.py").is_file(), "Push the new files, then use an updated checkout."
os.chdir(project)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, check=True)
print("Experiment:", project)


## 2. Install requirements and verify the GPU

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before training."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


## 3. Locate scripts and run small offline correctness checks

In [ ]:
scripts = {
    "prepare": project / "1_prepare_assets.py",
    "train": project / "2_train.py",
    "experiment": project / "3_run_experiments.py",
    "analyze": project / "4_analyze.py",
    "predict": project / "5_predict.py",
}
for name, path in scripts.items():
    assert path.is_file(), path
    print(name, "->", path.name)

def run_script(name, *arguments):
    subprocess.run([sys.executable, "-u", str(scripts[name]), *map(str, arguments)], cwd=project, check=True)

subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=project, check=True)


## 4. Mount Drive and choose settings

Each update uses `20 × BATCH_PER_TASK` images. Both methods see the same sampled
images and augmentation draws. The default six runs use 10,000 steps each; measure
the smoke/pilot first before choosing a final budget. Validation metrics print
every 100 steps and at the final step. No runtime estimate is assumed.

Keep `RESULTS_FOLDER = "runs"` for condition folders directly under `/runs`.
If settings/code change at an existing method/step/seed, choose a new results
folder such as `runs_v2`; incompatible files are never overwritten. Resume only
with unchanged code, settings, device type, and prepared data. Colab reset removes
local assets, which will be downloaded again; it does not remove Drive results.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ESTR3114_CIFAR100")
SPLIT_SEED = 123
VALIDATION_PER_CLASS = 50
resources = Path("/content/ESTR3114_CIFAR100_assets") / f"split{SPLIT_SEED}_val{VALIDATION_PER_CLASS}"
assert not resources.resolve().is_relative_to(Path("/content/drive").resolve())

RESULTS_FOLDER = "runs"
ANALYSIS_TAG = "main_v1"
SMOKE_TAG = "smoke_v1"
STEPS = 10000
SEEDS = [42, 43, 44]
BATCH_PER_TASK = 8
EVAL_BATCH_SIZE = 128
LEARNING_RATE = 0.01
EVAL_EVERY = 100
DIAGNOSTICS_EVERY = 100
SAVE_MODELS = True
RESUME = True

results = DRIVE_ROOT / RESULTS_FOLDER
analysis = DRIVE_ROOT / "analysis" / ANALYSIS_TAG
smoke_root = DRIVE_ROOT / "smoke" / SMOKE_TAG
results.mkdir(parents=True, exist_ok=True)
print("Local assets:", resources)
print("Main condition folders directly under:", results)
print("Analysis:", analysis)
print("Conditions:", 2 * len(set(SEEDS)))
print("Images per update:", 20 * BATCH_PER_TASK)
print("Equivalent training epochs:", round(STEPS * 20 * BATCH_PER_TASK / (50000 - 100 * VALIDATION_PER_CLASS), 2))


## 5. Prepare independent local CIFAR-100 assets

The official archive is verified before reading labels. Existing complete assets
are verified and reused. Train/validation indices and channel statistics use only
the original training set; the official test split is retained. This cell never
writes the dataset or cache into Google Drive.

In [ ]:
run_script("prepare", "--resources", resources,
           "--split-seed", SPLIT_SEED, "--validation-per-class", VALIDATION_PER_CLASS)
manifest = json.loads((resources / "manifest.json").read_text())
mapping = json.loads((resources / "task_mapping.json").read_text())
print(json.dumps(manifest, indent=2))
for task in mapping["tasks"]:
    print(task["id"], task["name"], "->", ", ".join(task["class_names"]))


## 6. Two-step GPU smoke test

This checks the full-sized CNN and actual prepared data with both methods. It
does not evaluate test images. Checkpoints are saved so an interruption can
resume, and smoke outputs stay outside main results. If an incompatible old
smoke folder exists after a settings change, choose a fresh `SMOKE_TAG`.

In [ ]:
for method in ("joint", "pcgrad"):
    run_script("train", "--method", method, "--seed", SEEDS[0],
               "--steps", 2, "--batch-per-task", 1, "--eval-batch-size", EVAL_BATCH_SIZE,
               "--eval-every", 2, "--diagnostics-every", 1,
               "--device", "cuda", "--resources", resources,
               "--save-model", "--resume", "--output", smoke_root / method)


## 7. Run the matched main experiment

This launches two methods for every seed and may take substantial time. Each
trainer prints validation results every 100 steps. Final/best-validation/last
checkpoints and per-class metrics persist on Drive. Compatible completed runs
are skipped; an incomplete run resumes from its last committed checkpoint when
`SAVE_MODELS` and `RESUME` are enabled. The main experiment does not inspect test
performance. Start with a shorter pilot or fewer seeds if desired, using a
separate results folder whenever other settings change.

In [ ]:
arguments = ["--seeds", *SEEDS, "--steps", STEPS,
             "--batch-per-task", BATCH_PER_TASK, "--eval-batch-size", EVAL_BATCH_SIZE,
             "--learning-rate", LEARNING_RATE,
             "--eval-every", EVAL_EVERY, "--diagnostics-every", DIAGNOSTICS_EVERY,
             "--device", "cuda", "--resources", resources, "--output", results]
if SAVE_MODELS:
    arguments.append("--save-model")
if RESUME and SAVE_MODELS:
    arguments.append("--resume")
run_script("experiment", *arguments)


## 8. Analyze validation results and show figures (GPU not required)

In [ ]:
run_script("analyze", "--results", results, "--output", analysis, "--steps", STEPS)
from IPython.display import display, Image, Markdown
display(Markdown((analysis / "report.md").read_text()))
for figure in sorted(analysis.glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure)))
print("Per-class metrics:", analysis / "per_class_metrics.csv")
print("Raw confusion matrices and task-pair cosine matrices are retained as JSON.")


## 9. Optional held-out test evaluation after validation inspection

Enable `EVALUATE_TEST` only after choosing your configuration from validation.
This evaluates saved **final and best-validation** checkpoints without retraining.
Their separate files contain per-task/class F1, confusion counts, and per-image
probabilities. Existing matching evaluations are reused. Test scores never select
the best checkpoint. Re-run analysis to include these new evaluation files.

In [ ]:
EVALUATE_TEST = False
if EVALUATE_TEST:
    import hashlib
    def checkpoint_hash(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    count = 0
    for method in ("joint", "pcgrad"):
        for seed in dict.fromkeys(SEEDS):
            run = results / f"{method}_step{STEPS}_seed{seed}"
            for label in ("final", "best_validation"):
                checkpoint = run / "checkpoints" / f"{label}.pt"
                if not checkpoint.is_file():
                    raise FileNotFoundError(f"No saved checkpoint: {checkpoint}; train with SAVE_MODELS=True first.")
                output = run / f"evaluation_test_{label}.json"
                if output.is_file():
                    prior = json.loads(output.read_text())
                    assert prior["checkpoint_sha256"] == checkpoint_hash(checkpoint), "Existing evaluation belongs to different weights."
                    print("Reusing:", output.name, run.name)
                else:
                    run_script("predict", "--checkpoint", checkpoint, "--evaluate-split", "test",
                               "--resources", resources, "--device", "cuda",
                               "--batch-size", EVAL_BATCH_SIZE, "--output", output)
                count += 1
    print("Evaluated/reused checkpoint states:", count)
    run_script("analyze", "--results", results, "--output", analysis, "--steps", STEPS)
    display(Markdown((analysis / "report.md").read_text()))
else:
    print("Test evaluation is disabled; use validation for configuration selection.")


## 10. Optional classification of an uploaded image

The superclass must be supplied. Set `PREDICT_IMAGE=True` to upload one image;
otherwise this cell skips under Run all. Set `TASK` to an ID or exact superclass
name printed in step 5. The selected checkpoint contains its own mapping and
normalization; no dataset is needed for this prediction.

In [ ]:
PREDICT_IMAGE = False
TASK = "fish"
PREDICT_METHOD = "pcgrad"
if PREDICT_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        filename = next(iter(uploaded))
        checkpoint = results / f"{PREDICT_METHOD}_step{STEPS}_seed{SEEDS[0]}" / "checkpoints" / "best_validation.pt"
        run_script("predict", "--checkpoint", checkpoint, "--image", Path(filename).resolve(),
                   "--task", TASK, "--device", "cuda")


## 11. Optional download of metrics and figures

Outputs already persist in Drive. This ZIP includes logs, settings, metrics, raw
confusion data, and plots; it **excludes model checkpoints and dataset assets**.
Enable the switch to download it. Step 12 downloads model files separately.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import files
    archive = Path("/content/ESTR3114_CIFAR100_results.zip")
    with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
        for label, base in (("results", results), ("analysis", analysis)):
            if not base.exists():
                continue
            for path in base.rglob("*"):
                if path.is_file() and path.suffix in {".json", ".jsonl", ".csv", ".png", ".md"}:
                    bundle.write(path, str(Path(label) / path.relative_to(base)))
    files.download(str(archive))
else:
    print("Outputs remain in Drive:", results, analysis)


## 12. Download saved models

Enable `DOWNLOAD_MODELS` to download final and best-validation checkpoints from
the main runs. They include model/optimizer/RNG state and label metadata. `last.pt`
is omitted to avoid another duplicate state. Choose one `MODEL_RUN` to reduce ZIP
size, or leave it as `None` for all current-budget main runs. The ZIP is built on
Colab's local disk with no tensor compression; assets are never included.

In [ ]:
DOWNLOAD_MODELS = False
MODEL_RUN = None  # e.g. "pcgrad_step10000_seed42"; None means all configured main runs
if DOWNLOAD_MODELS:
    from zipfile import ZipFile, ZIP_STORED
    from google.colab import files
    run_folders = (
        [results / MODEL_RUN] if MODEL_RUN is not None else
        [results / f"{method}_step{STEPS}_seed{seed}"
         for method in ("joint", "pcgrad") for seed in dict.fromkeys(SEEDS)]
    )
    checkpoints = [run / "checkpoints" / filename for run in run_folders
                   for filename in ("final.pt", "best_validation.pt")
                   if (run / "checkpoints" / filename).is_file()]
    if not checkpoints:
        print("No saved models. Train with SAVE_MODELS=True first.")
    else:
        archive = Path("/content/ESTR3114_CIFAR100_models.zip")
        with ZipFile(archive, "w", compression=ZIP_STORED, allowZip64=True) as bundle:
            for checkpoint in checkpoints:
                bundle.write(checkpoint, str(checkpoint.relative_to(results)))
        print("Downloading checkpoint files:", len(checkpoints))
        files.download(str(archive))
else:
    print("Set DOWNLOAD_MODELS=True to download saved model weights.")
